# Atividade 01 — Redes Neurais (Perceptron e MLP)

**Tema:** classificação de dois perfis de clientes de um app de delivery.

O Perceptron resolve problemas linearmente separáveis, mas falha em padrões não lineares. Uma MLP (rede com camada oculta) consegue separar os grupos.

> Rode todas as células em ordem no Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Dataset sintético: dois grupos em forma de lua (não linearmente separáveis)
X, y = make_moons(n_samples=200, noise=0.20, random_state=42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=40)
plt.title("Perfis de clientes — Grupo A (0) vs Grupo B (1)")
plt.xlabel("Gasto médio semanal")
plt.ylabel("Frequência de pedidos")
plt.show()

print(f"Amostras: {len(y)} | Treino: {len(y_treino)} | Teste: {len(y_teste)}")

In [ ]:
def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te):
    modelo.fit(X_tr, y_tr)
    pred_tr = modelo.predict(X_tr)
    pred_te = modelo.predict(X_te)
    acc_tr = accuracy_score(y_tr, pred_tr)
    acc_te = accuracy_score(y_te, pred_te)
    print(f"\n=== {nome} ===")
    print(f"Acurácia treino: {acc_tr:.2%}")
    print(f"Acurácia teste:  {acc_te:.2%}")
    return modelo, acc_te

In [ ]:
perceptron = Perceptron(max_iter=1000, eta0=0.1, random_state=42)
perceptron, acc_perc = avaliar_modelo(
    "Perceptron — perfis de clientes", perceptron, X_treino, y_treino, X_teste, y_teste
)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation="relu",
    solver="adam",
    max_iter=5000,
    random_state=42
)
mlp, acc_mlp = avaliar_modelo(
    "MLP — perfis de clientes", mlp, X_treino, y_treino, X_teste, y_teste
)

In [ ]:
pd.DataFrame({
    "Modelo": ["Perceptron", "MLP"],
    "Acurácia no teste": [acc_perc, acc_mlp]
})

In [ ]:
print("=== Arquitetura da MLP ===")
print("Camadas:", mlp.n_layers_)
print("Entradas:", mlp.n_features_in_)
print("Ocultas:", mlp.hidden_layer_sizes)
print("Saídas:", mlp.n_outputs_)

for i, (W, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_)):
    print(f"\nCamada {i} -> {i+1} | pesos {W.shape} | bias {b.shape}")

In [ ]:
W1 = mlp.coefs_[0]
b1 = mlp.intercepts_[0]

display(pd.DataFrame(W1, index=["gasto", "frequencia"],
                       columns=[f"h{i+1}" for i in range(W1.shape[1])]))
display(pd.DataFrame([b1], index=["bias"],
                       columns=[f"h{i+1}" for i in range(len(b1))]))

In [ ]:
def desenhar_rede_mlp(modelo, titulo="Rede MLP"):
    camadas = [modelo.n_features_in_]
    if isinstance(modelo.hidden_layer_sizes, int):
        camadas.append(modelo.hidden_layer_sizes)
    else:
        camadas.extend(modelo.hidden_layer_sizes)
    camadas.append(modelo.n_outputs_)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")
    posicoes = []
    for idx, n in enumerate(camadas):
        ys = [(n - 1) / 2 - i for i in range(n)]
        posicoes.append(ys)
        for i, y in enumerate(ys):
            if idx == 0:
                rot = ["gasto", "freq"][i] if n == 2 else f"x{i+1}"
            elif idx == len(camadas) - 1:
                rot = "grupo"
            else:
                rot = f"h{i+1}"
            ax.add_patch(plt.Circle((idx, y), 0.12, fill=False, linewidth=2))
            ax.text(idx, y, rot, ha="center", va="center", fontsize=9)
    for camada_idx, W in enumerate(modelo.coefs_):
        for i in range(W.shape[0]):
            for j in range(W.shape[1]):
                ax.plot([camada_idx, camada_idx + 1],
                        [posicoes[camada_idx][i], posicoes[camada_idx + 1][j]], linewidth=0.8)
    ax.set_title(titulo)
    plt.show()

desenhar_rede_mlp(mlp, "MLP treinada — classificação de perfis de clientes")

In [ ]:
probs = mlp.predict_proba(X_teste[:8])
preds = mlp.predict(X_teste[:8])
tabela = pd.DataFrame(probs, columns=["P(Grupo A)", "P(Grupo B)"])
tabela["Previsto"] = preds
tabela["Real"] = y_teste[:8]
tabela

## Conclusão

O **Perceptron** é um classificador linear e tende a ter baixa acurácia quando os grupos não são separáveis por uma reta. A **MLP** adiciona camadas ocultas e consegue modelar fronteiras não lineares, classificando melhor os perfis de clientes.